# Day 005 · 收益率的几种算法
**Returns Math** · 阶段 P1 · 量化基础

> 这是一节看似简单实则关键的课。如果你连'算收益率'都没弄准,后面所有的 Sharpe、夏普、年化都是错的。今天我们把简单收益、对数收益、年化收益讲透,顺便揭穿一个所有人(包括很多基金经理)都犯的小错。今天的内容看起来像数学,实际是量化的'地基':地基不平,楼盖再高也歪。

---

**课件生成日期:** 2026-05-01  ·  **建议学习时长:** 18 分钟

学习路径建议:1)先看视频建立直觉 → 2)阅读本 notebook 跑代码 → 3)看 PDF 课件复习要点 → 4)做自测题

## 🔧 第一步:环境自检 + 自动安装

**第一次拿到这份 notebook,请先运行下面这一格。** 它会:
1. 检查所有需要的 Python 包,缺什么自动 `pip install` 装上
2. 注入中文字体到 matplotlib(让图标不出乱码)
3. 跑完看到 `✓ 环境就绪` 就可以继续下面的代码

> 这一格只在第一次跑要等几十秒,后面再开 notebook 就秒过。

In [ ]:
# === 环境自检 + 自动安装(运行此单元格即可) ===
# 检测缺失的库 → 自动 pip 安装 → 注入中文字体 → 一行命令搞定
import importlib
import subprocess
import sys
import os

REQUIRED = ["matplotlib", "numpy", "numpy_financial", "pandas", "scipy", "sklearn", "statsmodels", "yfinance"]
PIP_NAME = {
    "sklearn": "scikit-learn",
    "cv2": "opencv-python",
    "PIL": "Pillow",
    "bs4": "beautifulsoup4",
    "yaml": "PyYAML",
}

missing = []
for mod in REQUIRED:
    try:
        importlib.import_module(mod)
    except ImportError:
        missing.append(PIP_NAME.get(mod, mod))

if missing:
    print(f"⏳ 缺少 {len(missing)} 个包,正在自动安装:{missing}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    print("✓ 安装完成")
else:
    print(f"✓ 所有 {len(REQUIRED)} 个必需库已就绪")

# === 中文字体配置(让 matplotlib 不出乱码) ===
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

CJK_FONT_PATHS = [
    "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",  # Linux/WSL
    "C:/Windows/Fonts/msyh.ttc",                               # Windows 微软雅黑
    "C:/Windows/Fonts/simhei.ttf",                             # Windows 黑体
    "/System/Library/Fonts/PingFang.ttc",                      # macOS 苹方
    "/System/Library/Fonts/STHeiti Medium.ttc",                # macOS 黑体
]
for p in CJK_FONT_PATHS:
    if os.path.exists(p):
        fm.fontManager.addfont(p)
        print(f"✓ 中文字体已加载:{os.path.basename(p)}")
        break
plt.rcParams["font.sans-serif"] = ["Noto Sans CJK JP", "Microsoft YaHei",
                                    "PingFang SC", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
print("✓ 环境就绪 — 现在可以跑下面的代码单元格")


## 学习目标

- 区分简单收益和对数收益的数学差别 + 各自的适用场景
- 学会正确做时间窗口的年化(知道为什么要 √252 不是 365)
- 看懂为什么对数收益更适合量化建模 + 计算夏普
- 处理收益率的常见问题:停牌 / 缺失值 / 异常值 / 复权
- 用 Python 实现 5 种收益率算法并跑出来对比

## 历史背景:Bachelier、Black-Scholes 和一个真实的算错案例

1900 年,法国数学家 Louis Bachelier 在博士论文《投机理论》中第一次用对数收益建模股价随机游走——他比 Einstein 早 5 年用了同样的数学。论文当年被忽略,但 60 年后被 Samuelson 重新发现,成为整个现代金融数学的源头。

1973 年 Black-Scholes-Merton 期权定价公式发表,核心假设之一就是'对数收益服从正态分布'。这个假设深刻影响了之后 50 年的量化金融。

2018 年某只 A 股私募发布业绩报告称'近 1 年累计收益 50%',监管核查发现:他们用了月度简单收益 × 12 作为年化,这是错的。正确的算法是 (1+月收益)^12 - 1。一个简单收益的算法错误,让真实年化从 30% 被'美化'成 50%。这种错误在散户的雪球/微博晒单里更常见,看到时多用一个心眼。

本节最重要的一句话:**简单收益不可加,对数收益可加**。这一句话决定了你接下来 365 天所有的统计计算用哪种。

**关键人物:**
- Louis Bachelier(1900,对数收益开山)
- Fischer Black、Myron Scholes、Robert Merton(1973,BS 公式)
- Paul Samuelson(诺奖,把对数收益带回现代金融)

## 核心概念

下面每一节是听完视频后回头细读的内容。

### 1. 简单收益 vs 对数收益:数学差别

简单收益(算术收益):R_simple = (P_t - P_{t-1}) / P_{t-1}。这是最直观的算法,'今天比昨天涨了 2%'。

对数收益:R_log = ln(P_t / P_{t-1}) = ln(1 + R_simple)。把价格变化取对数。

关键区别:
- 短期(< 5%)两者几乎一样:ln(1.01) ≈ 0.01
- 长期或大幅波动:差别明显。涨 50%,简单收益 = 0.5,对数收益 = 0.405
- 简单收益不可加:今天 +50% 明天 -50% ≠ 持平,实际是 -25%(1.5 × 0.5 = 0.75)
- 对数收益可加:ln(1.5) + ln(0.5) = 0.405 - 0.693 = -0.288,exp(-0.288) = 0.75 ✓

所以做时间序列建模、夏普计算、统计推断时必须用对数收益。简单收益只用于面向人的报表展示。

```
R_log_t = ln(P_t) - ln(P_{t-1});  Σ R_log = ln(P_T / P_0)
```

> **举例:** 腾讯 2021 年从 766 跌到 224,简单跌幅 -71%。对数收益 = ln(224/766) = -1.23。前者直观,后者方便和其它周期累加。


### 2. 为什么是 √252,不是 √365

年化波动率公式:σ_annual = σ_daily × √N,其中 N 是一年的交易日数。

这里 N 用 252 而不是 365,因为周末和节假日股市关闭——金融时间和日历时间不是一回事。一年大约有 252 个交易日(美股),A 股大约 244,港股 247,精确数字按市场调整。

为什么是 √N 不是 N?因为方差(variance)线性可加,但波动率(标准差 = √方差)是平方根关系。这是中心极限定理的直接结果,不是经验法则。

对收益的年化是另一回事:R_annual = (1 + R_daily)^N - 1。注意这里用的是 N(连乘),不是 √N。波动率 √N,收益 N,记住这个区别。

Kelly 公式、Sharpe 计算、风险预算全都依赖这两个年化方法的正确使用。

```
σ_ann = σ_daily × √252;  R_ann = (1 + R_daily_avg)^252 - 1
```

> **举例:** 茅台日均收益 0.05%,日波动率 1.8%。年化收益 ≈ 0.05% × 252 = 12.6%。年化波动率 ≈ 1.8% × √252 = 28.6%。Sharpe ≈ 12.6 / 28.6 ≈ 0.44。


### 3. 总收益 vs 价格收益(分红的处理)

股价的变动有两个来源:价格波动 + 分红。

价格收益(price return):只看 P_t 和 P_{t-1} 的差,不算分红
总收益(total return):假设分红立即再投资,P 在分红当天'人为补回'

两者长期差距大:
- 美股 SP500:1990-2024 价格年化 ~7.5%,总收益年化 ~10.5%。差 3% 来自分红
- A 股大蓝筹:工商银行 10 年价格涨 30%,总收益涨 110%。分红贡献 80%

yfinance/Tushare 提供的复权价已经包含了分红再投资,这是'后复权'。前复权 = 总收益线被压平到今天。回测必须用复权价(后复权或前复权都行,但 retrieved 一致)。

警告:有些公开业绩展示用'价格涨幅'代替'总收益'来藏掉低分红表现,看雪球晒单时多个心眼。

> **举例:** 工商银行 2014-2024:价格从 3.5 涨到 5.5(+57%),总收益(含分红再投)从 3.5 涨到 11.5(+229%)。如果只看价格,你以为这只股 10 年很弱,实际它是高分红的赚钱机器。


### 4. 累计收益 vs 期间收益(展示的两种方式)

累计收益(cumulative return):从起点到当前的总变化。常用画法:净值曲线起点 = 1,曲线 = (1 + R_1) × (1 + R_2) × ...

期间收益(periodic return):某个固定窗口的收益,比如月度收益、季度收益。常用画法:柱状图。

展示侧重不同:
- 累计:显示长期复合效应,适合营销和长期跟踪
- 期间:显示稳定性和最近的状态,适合复盘和波动管理

实际报表两个都要看。累计漂亮但忽视回撤,期间真实但缺长期感。Pyfolio/QuantStats 等工具会自动出两套图。

小坑:期间收益取均值不能直接代表年化。日均收益 0.04% × 252 = 10.08% 不等于真实年化(因为复利)。真实年化 = (1 + 日均)^252 - 1 ≈ 10.6%。

> **举例:** 某基金 2024 年 Q1: 月收益 +5%, +5%, -8%。累计 = 1.05×1.05×0.92 - 1 = +1.4%。简单加 = +2%。差别不大但月份越多差距越显著。


### 5. 实战:5 种收益率算法对比

完整工具箱(配套代码会全部跑一遍):

1. 简单收益:pct_change(),展示给客户用
2. 对数收益:np.log(P/P.shift()),建模/Sharpe 用
3. 累计简单:(1+R).cumprod() - 1,画净值曲线
4. 累计对数:R.cumsum(),画对数净值曲线
5. 年化:(1+R)^N - 1 收益,σ × √N 波动

常见错误清单:
- 月度简单收益直接 × 12 当年化 ❌
- 用价格收益计算 Sharpe(没考虑分红)❌
- 短窗口(< 60 日)硬年化(噪声放大) ❌
- 加密 24h 用 252 (应该用 365 或 8760 小时) ❌
- 复权和不复权数据混用 ❌

> **举例:** QuantStats 一行代码:qs.reports.full(returns) 自动产出 30+ 指标 + 累计/期间双视图,内部全部用对数收益做计算,简单收益做展示。是开源工具的最佳实践。


## 实操:5 种收益率算法 + 跨四市场五资产实战

下面这段代码跟视频里讲解的 highlights 是一致的,可以**直接 Run All** 看结果。

**依赖安装:**
```bash
pip install pandas numpy matplotlib yfinance akshare statsmodels
```


In [ ]:
# day_005_returns_math.py — 招行/美团/微软/GLD/BTC 五资产收益率全套
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt

tickers = {
    '招商银行 A':  '600036.SS',
    '美团 H':      '3690.HK',
    'MSFT US':     'MSFT',
    '黄金 ETF':    'GLD',
    '比特币':      'BTC-USD',
}
df = yf.download(list(tickers.values()), period='3y', auto_adjust=True, progress=False)['Close']
df.columns = list(tickers.keys())
df = df.dropna()

simple_ret = df.pct_change().dropna()
log_ret = np.log(df / df.shift(1)).dropna()
cum_simple = (1 + simple_ret).cumprod() - 1
cum_log = log_ret.cumsum()

ann_ret = (1 + simple_ret.mean())**252 - 1
ann_vol = simple_ret.std() * np.sqrt(252)
sharpe = ann_ret / ann_vol
noise_ratio = ann_ret / ann_vol

result = pd.DataFrame({
    '3年累计简单(%)': (cum_simple.iloc[-1] * 100).round(1),
    '3年累计对数(%)': (cum_log.iloc[-1] * 100).round(1),
    '年化收益(%)':   (ann_ret * 100).round(1),
    '年化波动(%)':   (ann_vol * 100).round(1),
    'Sharpe':         sharpe.round(2),
})
print(result.to_string())

monthly = simple_ret.resample('ME').apply(lambda x: (1+x).prod() - 1)
wrong_ann = monthly.mean() * 12
right_ann = (1 + monthly.mean())**12 - 1
print('\n[陷阱演示] 月平均×12 vs 正确年化')
for name in monthly.columns:
    print(f'  {name}: 错误 {wrong_ann[name]*100:+.1f}% / 正确 {right_ann[name]*100:+.1f}%')

(1 + simple_ret).cumprod().plot(figsize=(12, 5), title='3 年累计净值')
plt.tight_layout(); plt.savefig('day005_cumret.png', dpi=120)

## 真实市场案例

| 市场 | 标的 | 实战观察 |
| --- | --- | --- |
| A 股 | 600519 茅台 | 2019-2024 价格涨 ~150%,后复权(含分红)涨 ~170%。年化波动 28%,Sharpe ~0.45。简单收益 vs 对数:5 年差距 ~5pp。 |
| 港股 | 0700 腾讯 | 2019-2024 累计 +5%(看似平),但中间 -70% 又 +60%,真实波动巨大。仅看累计漏掉风险,必须看期间分布。 |
| 美股 | SPY | 2019-2024 价格年化 ~10%,总收益年化 ~12%。差距 2pp 来自分红。年化波动 18%,Sharpe ~0.55。 |
| 加密 | BTC-USD | 2019-2024 累计 +800%(7 倍),年化波动 65%。年化用 252 还是 365 是争议——传统倾向 365(因为 24/7),但许多研究为可比性仍用 252。 |


## 常见坑

### ⚠ 01. 不区分简单和对数,混算 Sharpe

用简单日收益的均值和标准差直接算 Sharpe → 数学不对(简单收益不可加)。正确:用对数收益算 Sharpe,简单收益算累计展示。这两个用法分清楚。

### ⚠ 02. 短期年化

用 1 个月数据 × 12 推年化,或者 3 个月 × 4。短窗口噪声大,年化数字毫无意义。规则:年化的最小窗口是 1 年(252 个交易日),低于此只能讲'近期表现'。

### ⚠ 03. 忘了分红

用价格收益算 Sharpe → 低估真实回报。尤其是高分红蓝筹股(银行/电力/REITs),分红贡献能占 50% 以上。永远用复权价。

### ⚠ 04. 复权数据用错

前复权和后复权混用,导致同一只股票不同时点的'同一价格'其实代表不同收益。规则:回测期内只用一种复权(常用后复权),数据更新后整组重算。

### ⚠ 05. 加密 24h 没跨节假日

用 252 算年化波动率会低估,因为加密市场没有周末关闭。正确:用 365 或者 8760 小时(BTC 经常按小时算 vol)。这个错误在加密研报里非常常见。

## 实战 SOP · 收益率使用 SOP

1. 面向人的报表展示:用简单收益(直观)+ 累计净值线
2. 做统计推断 / 建模 / 夏普计算:用对数收益
3. 时间窗口年化:收益用 (1+R)^N - 1,波动率用 σ × √N,N 不是 365 是交易日数
4. 美股 N=252,A 股 N=244,港股 N=247,加密 N=365 或更细
5. 数据源固定后,'用同一种复权'(常推荐后复权)做整套研究
6. 短窗口(< 60 日)绝对不年化,只讲'近期表现'
7. QuantStats / pyfolio 是行业最佳实践工具,直接用比自己造轮子靠谱

> 把这段打印贴在你电脑边,执行 1000 次它会回报你。

## 总结 · 你应该带走的

2. 简单收益直观但不可加;对数收益可加,适合统计建模和 Sharpe 计算。
3. 年化:收益用 (1+R)^N - 1,波动率用 σ × √N。注意 N 是交易日数(美 252、A 股 244)。
4. 总收益(含分红再投)长期比价格收益高得多,蓝筹股差距尤其明显。
5. 永远用复权价,前后复权选一种用整套,不混用。
6. 短窗口(< 60 日)不要硬年化,数学上没意义,误导别人也误导自己。
7. QuantStats / Pyfolio 已经把所有这些算法封装好,推荐直接用。
8. 看到'月平均 × 12'当年化的报告,提高警惕——这是常见的(有意或无意)误导。
9. 今天的内容是地基,做不好后面的 Sharpe/Calmar/夏普全是错的。打好。

## 自测题

**Q1.** 你能用 30 秒解释:为什么算 Sharpe 要用对数收益,而不是简单收益?
**Q2.** 茅台过去一个月日均收益 +0.5%,日波动率 2%。它的年化收益和年化波动大约是多少?Sharpe?
**Q3.** 看到一个雪球大V晒'近 6 个月赚 30%,年化 60%',你的第一反应应该是什么?

把答案写下来,3 天后再回看。

## 下一节预告

**Day 006 · 复利与贴现** (Compounding & Discounting)

Day 6:复利与贴现 — 时间价值是金融的灵魂。我们用 Python 实现 NPV、IRR、连续复利,并看懂为什么巴菲特说'复利是世界第八大奇迹'。

## 推荐阅读

- 《Active Portfolio Management》Grinold & Kahn 第 2 章 — 收益率基础(行业经典)
- QuantStats GitHub README + examples — 业内最常用的开源指标库,直接看代码
- Lopez de Prado《Advances in Financial Machine Learning》第 2 章 — 收益率序列处理细节
- Tushare 的'复权数据接口文档' — A 股复权计算的具体实现
- yfinance README 'auto_adjust' 参数说明 — 美股/港股复权的最常见错误来源